# Thursday: prepared demonstrations
Run top to bottom in Jupyter. Core examples use only the Python standard library. No API keys, remote requests or GPU required. Cell outputs are intentionally cleared for live teaching. The separate Thursday_Demo_Reader.html contains saved results. Timings are observations on this machine, not universal speed ratios.

Core route: A3 → B1/B2 → C → D. A1/A2 are optional during algorithms. Each section includes a prediction before execution.

## A1. Search traces — optional
Predict the comparisons needed to find 42 in the same sorted list shown on slides 14–15.

In [ ]:
values = [3, 7, 11, 14, 18, 21, 26, 31, 35, 42, 48, 53, 60, 66, 72, 80]

def linear_trace(values, target):
    seen = []
    for i, value in enumerate(values):
        seen.append(value)
        if value == target:
            return i, seen
    return None, seen

def binary_trace(values, target):
    lo, hi = 0, len(values) - 1
    seen = []
    while lo <= hi:
        mid = (lo + hi) // 2
        seen.append(values[mid])
        if values[mid] == target:
            return mid, seen
        if values[mid] < target:
            lo = mid + 1
        else:
            hi = mid - 1
    return None, seen

linear_index, linear_seen = linear_trace(values, 42)
binary_index, binary_seen = binary_trace(values, 42)
assert linear_index == binary_index == 9
assert linear_seen == values[:10]
assert binary_seen == [31, 53, 42]
print('Linear:', len(linear_seen), 'comparisons', linear_seen)
print('Binary:', len(binary_seen), 'comparisons', binary_seen)


The trace lists are extra instrumentation and grow with the trace length. Search without these lists needs only a few variables. Binary search assumes the input is already sorted.

## A2. Build cost and repeated queries — optional
Predict when the cost of building a set is worthwhile. Include that cost in the comparison. We use identical, absent integer keys so list membership scans the full list. No file, network or API calls occur inside the measurement.

In [ ]:
from statistics import median
from timeit import repeat

def list_queries(items, queries):
    return [query in items for query in queries]

def build_and_query(items, queries):
    lookup = set(items)
    return [query in lookup for query in queries]

def milliseconds(function):
    return 1000 * median(repeat(function, number=1, repeat=5))

print(f"{'n':>7} {'q':>6} {'list ms':>12} {'build+query ms':>16} {'set query ms':>14}")
for n in (1_000, 10_000):
    items = list(range(n))
    for q in (1, 1_000):
        queries = list(range(n, n + q))
        lookup = set(items)
        expected = [False] * q
        assert list_queries(items, queries) == build_and_query(items, queries) == expected
        assert [query in lookup for query in queries] == expected
        a = milliseconds(lambda: list_queries(items, queries))
        b = milliseconds(lambda: build_and_query(items, queries))
        c = milliseconds(lambda: [query in lookup for query in queries])
        print(f'{n:7,d} {q:6,d} {a:12.4f} {b:16.4f} {c:14.4f}')


Discuss the end-to-end column first. Query-only timing assumes the set already exists. The set adds O(n) auxiliary storage; both versions also retain the same O(q) output list here. Average/expected hash costs assume ordinary hashing behavior. These timings include result construction and interpreter overhead.

## 3. Recursion and iteration
The functions match slides 11–12. Predict the live calls for [2, 4, 6]. No slicing is used.

In [ ]:
def recursive_total(values, i=0):
    if i == len(values):
        return 0
    return values[i] + recursive_total(values, i + 1)

def iterative_total(values):
    result = 0
    for value in values:
        result += value
    return result

for example in ([], [2, 4, 6], [-3, 5, 9]):
    expected = sum(example)
    assert recursive_total(example) == iterative_total(example) == expected
    print(example, '=>', expected)


For [2, 4, 6], four calls are live at maximum depth, including the base case. Both approaches visit all n values. Recursion retains O(n) frames; iteration uses O(1) extra space under the usual unit-cost arithmetic model. Input storage is excluded. Do not raise the recursion limit as a substitute for choosing an iterative approach for long lists.

## A3. Generators: when does work happen?
Predict what prints when `values()` is called, then at each `next`. Calling a generator function does not start its body.

Slides 26–28. Core demo: about 4 minutes.

In [ ]:
def values():
    for value in (2, 4, 6):
        print('producing', value)
        yield value

g = values()
print('generator created')
assert next(g) == 2
assert next(g) == 4
assert next(g) == 6
assert list(g) == []
print('generator exhausted')

### Same answer, different retained intermediates
Predict which peak grows as n grows. Both versions do O(n) arithmetic under a fixed-size arithmetic model. `range` avoids a preloaded input list. `tracemalloc` measures traced Python allocations, not total process RAM. Runtime is measured separately from memory tracing.

In [ ]:
import gc, tracemalloc
from time import perf_counter

def materialized(n):
    return sum([x*x for x in range(n)])

def streamed(n):
    return sum(x*x for x in range(n))

print('n       method          peak KiB    time ms')
for n in (10_000, 100_000):
    expected = n*(n-1)*(2*n-1)//6
    for fn in (materialized, streamed):
        gc.collect()
        tracemalloc.start()
        answer = fn(n)
        _, peak = tracemalloc.get_traced_memory()
        tracemalloc.stop()
        start = perf_counter()
        check = fn(n)
        elapsed = perf_counter()-start
        assert answer == check == expected
        print(f'{n:<7} {fn.__name__:<14} {peak/1024:9.1f} {elapsed*1000:10.2f}')

Debrief: streaming reduces retained intermediate storage here; it does not remove the arithmetic or guarantee lower runtime. Consuming a generator into a list would retain those values again.

## B1. Overlapping waits, with a limit
Slides 38–41. Predict six independent 0.4-second waits with limits 1, 2 and 6: ideal elapsed times 2.4, 1.2 and 0.4 seconds. These are simulated waits, not actual quote retrieval. The semaphore limits in-flight operations; it is not a rate limiter or a bound on all queued task objects.

In [ ]:
import asyncio

async def quote_run(limit, fail_id=None):
    semaphore = asyncio.Semaphore(limit)
    active = peak_active = 0
    events = []
    epoch = perf_counter()
    async def fetch(i):
        nonlocal active, peak_active
        async with semaphore:
            active += 1
            peak_active = max(peak_active, active)
            start = perf_counter()-epoch
            try:
                await asyncio.sleep(0.4)
                if i == fail_id:
                    raise ValueError('simulated unavailable quote')
                return {'id': i, 'value': 100+i, 'error': None}
            except ValueError as exc:
                return {'id': i, 'value': None, 'error': str(exc)}
            finally:
                events.append((i, start, perf_counter()-epoch))
                active -= 1
    results = await asyncio.gather(*(fetch(i) for i in range(6)))
    elapsed = perf_counter()-epoch
    assert active == 0 and peak_active <= limit
    assert [r['id'] for r in results] == list(range(6))
    return results, elapsed, peak_active, sorted(events)

runs = {}
for limit in (1, 2, 6):
    results, elapsed, peak, events = await quote_run(limit)
    assert [r['value'] for r in results] == list(range(100, 106))
    assert all(r['error'] is None for r in results)
    runs[limit] = (results, elapsed, peak, events)
    print(f'limit={limit}: {elapsed:.2f}s, peak in flight={peak}, values={[r["value"] for r in results]}')
print('All three runs produced the same ordered results.')

### Inspect the timing, not just the total
Each row below is an operation. The measured interval is when that operation held a semaphore slot. Starts differ slightly because scheduling has overhead.

In [ ]:
for limit, (_, elapsed, peak, events) in runs.items():
    print(f'\nLimit {limit}')
    for i, start, end in events:
        print(f'  request {i}: {start:4.2f}s → {end:4.2f}s')

## B2. Account for a failed request
Predict how many outcomes we need if request 3 fails. Faster completion does not justify silently dropping an input. This example catches one expected application error per request; real cancellation, timeouts and retry policy require deliberate handling.

In [ ]:
results, elapsed, peak, events = await quote_run(2, fail_id=3)
assert len(results) == 6
assert sum(r['error'] is not None for r in results) == 1
assert results[3]['value'] is None and results[3]['error']
assert all(r['value'] == 100+r['id'] for r in results if r['error'] is None)
assert peak <= 2
for r in results:
    print(r)
print('CHECK: six inputs accounted for; five values and one explicit failure.')

## C. KV-cache byte budget
Slides 47–51. Use the accompanying interactive walkthrough for the token steps. This calculation models dense full-attention KV storage; it is not a live transformer benchmark.

Predict the effect of doubling context or active sequences. Assumptions: 32 layers, 8 KV heads, head dimension 128, 2 bytes/value; no prefix sharing, compression, offloading or sliding-window cap. Model weights and temporary buffers are excluded.

In [ ]:
def kv_bytes(tokens, sequences=1, layers=32, kv_heads=8, head_dim=128, bytes_per_value=2):
    return 2*layers*sequences*tokens*kv_heads*head_dim*bytes_per_value

assert kv_bytes(1) == 128*1024
assert kv_bytes(4096) == 512*1024**2
assert kv_bytes(8192) == 1024**3
assert kv_bytes(8192, 4) == 4*1024**3
for tokens, sequences in ((4096,1), (8192,1), (8192,4)):
    size = kv_bytes(tokens, sequences)
    print(f'{tokens:5} tokens × {sequences} sequences: {size/1024**2:7.0f} MiB ({size/1024**3:.2f} GiB)')
print('K and V give the leading factor of two. Cache growth does not change model weights.')

## D. Prepared tool call → execution → independent checks
Slides 54–55. This is an instructor-authored request, not output from a live model. Narrow contract: finite nonnegative shares and price, for a long-only position. Predict why returning a number is insufficient evidence of correctness.

In [ ]:
import math

def position_value(shares, price):
    return shares*price

def execute(request):
    if set(request) != {'name', 'arguments'} or request['name'] != 'position_value':
        raise ValueError('unsupported request')
    args = request['arguments']
    if set(args) != {'shares', 'price'}:
        raise ValueError('expected shares and price')
    for value in args.values():
        if isinstance(value, bool) or not isinstance(value, (int, float)) or not math.isfinite(value) or value < 0:
            raise ValueError('finite nonnegative numbers required')
    result = position_value(**args)
    if not math.isfinite(result):
        raise ValueError('result must be finite')
    return result

request = {'name': 'position_value', 'arguments': {'shares': 2, 'price': 100}}
print('Prepared request:', request)
print('Executed result:', execute(request))
assert execute(request) == 200

def candidate_bad(shares, price):
    return shares+price

cases = [(2,100,200), (0,100,0), (0.5,80,40)]
for fn in (candidate_bad, position_value):
    failures = []
    for shares, price, expected in cases:
        actual = fn(shares, price)
        if actual != expected:
            failures.append((shares, price, expected, actual))
    print(fn.__name__, ':', len(cases)-len(failures), '/', len(cases), 'checks passed')
    for shares, price, expected, actual in failures:
        print(f'  {shares} × {price}: expected {expected}, got {actual}')
    assert len(failures) == (3 if fn is candidate_bad else 0)

for invalid in (-1, float('nan'), float('inf'), True):
    try:
        execute({'name':'position_value','arguments':{'shares':invalid,'price':100}})
    except ValueError:
        pass
    else:
        raise AssertionError('invalid input accepted')
print('Input checks passed. These narrow checks do not establish general agent reliability.')

## Instructor wrap-up
A: same result, different retained storage. B: overlap independent waits, bound resources, account for every result. C: reuse prior attention state at a memory cost. D: independent expected results distinguish execution from correctness.

Sources: January Lesson 6 demo notebook (search/sort, async, memory profiling); Python documentation for generators, asyncio and tracemalloc; Hugging Face Transformers cache explanation. The KV walkthrough and narrow tool checks are tailored to this Thursday continuation.